# 06 — Reshaping & Combining Arrays
**Goal:** Change the shape of arrays and combine multiple arrays into one.

```
reshape      → same data, new shape
transpose    → swap axes
concatenate  → join along an existing axis
stack        → join along a NEW axis
split        → cut one array into pieces
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

## 1. reshape — same data, new shape

In [ ]:
# Flat array of 12 numbers → matrix 3×4 → 4×3
arr = np.arange(1, 13)
print('1D:', arr)

m34 = arr.reshape(3, 4)
print('\n3×4:')
print(m34)

m43 = arr.reshape(4, 3)
print('\n4×3:')
print(m43)

In [ ]:
# Use -1 to let NumPy infer one dimension
arr = np.arange(12)
print(arr.reshape(3, -1))    # -1 → 4 (12/3)
print(arr.reshape(-1, 4))    # -1 → 3 (12/4)

# Flatten back to 1D
matrix = arr.reshape(3, 4)
print(matrix.flatten())      # always returns a copy
print(matrix.ravel())        # returns a view when possible

In [ ]:
# 1D → column vector (shape (n,1)) — useful for broadcasting
v = np.array([100, 200, 300])
print('v shape:         ', v.shape)          # (3,)
print('v[:,None] shape: ', v[:, None].shape) # (3, 1)
print(v[:, None])

## 2. transpose — swap rows and columns

In [ ]:
# (days, channels) → (channels, days)
daily = np.array([
    [3200, 2800, 3500],   # organic: 3 days
    [2800, 2900, 2700],   # paid
    [3500, 3400, 3600],   # email
])
print('Original shape:', daily.shape)   # (3, 3)
print(daily)

print('\nTransposed shape:', daily.T.shape)  # (3, 3) in this case
print(daily.T)

In [ ]:
# More meaningful: (3 metrics) × (5 channels) → (5 channels) × (3 metrics)
funnel = np.array([
    [3200, 2800, 3500, 2600, 4100],
    [2400, 1960, 2800, 1820, 3280],
    [  96,   59,  158,   47,  156],
])
print('Original (metrics × channels):', funnel.shape)
print('Transposed (channels × metrics):', funnel.T.shape)
print(funnel.T)

## 3. concatenate — join along existing axis

In [ ]:
# Stack vertically (add rows) → axis=0
jan = np.array([[3200, 2800], [96, 59]])
feb = np.array([[3400, 2900], [102, 63]])

combined = np.concatenate([jan, feb], axis=0)  # stack rows
print('axis=0 (rows):')
print(combined)
print('shape:', combined.shape)

In [ ]:
# Stack horizontally (add columns) → axis=1
sessions = np.array([[3200], [2800], [3500]])         # (3,1)
activations = np.array([[96], [59], [158]])            # (3,1)

combined_cols = np.concatenate([sessions, activations], axis=1)
print('axis=1 (columns):')
print(combined_cols)
print('shape:', combined_cols.shape)   # (3, 2)

In [ ]:
# Shortcuts for common cases
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print(np.vstack([a, b]))   # vertical stack (axis=0)
print()
print(np.hstack([a, b]))   # horizontal stack (axis=1)

## 4. stack — create a new axis

In [ ]:
# Each array is (5,) — one per channel for one day
day1 = np.array([3200, 2800, 3500, 2600, 4100])
day2 = np.array([3100, 2900, 3400, 2500, 4000])
day3 = np.array([3300, 2700, 3600, 2700, 4200])

# Stack along axis=0 → (3, 5) matrix: rows=days, cols=channels
sessions_matrix = np.stack([day1, day2, day3], axis=0)
print('stack axis=0 → (days, channels):', sessions_matrix.shape)
print(sessions_matrix)

# Stack along axis=1 → (5, 3) matrix: rows=channels, cols=days
sessions_T = np.stack([day1, day2, day3], axis=1)
print('\nstack axis=1 → (channels, days):', sessions_T.shape)
print(sessions_T)

## 5. split — divide an array into pieces

In [ ]:
# Split 90-day array into 3 months of 30 days each
rng = np.random.default_rng(42)
full_quarter = rng.integers(500, 1500, size=90)

jan, feb, mar = np.split(full_quarter, 3)
print('Jan shape:', jan.shape)    # (30,)
print('Jan mean: ', jan.mean().round(1))
print('Feb mean: ', feb.mean().round(1))
print('Mar mean: ', mar.mean().round(1))

## 6. Real example — build a 3D funnel tensor

In [ ]:
df = pd.read_csv('data/raw/funnel_data.csv')
channels_list = ['organic', 'paid', 'email', 'social', 'direct']
steps = ['visita_landing', 'inicio_solicitud', 'activacion_tarjeta']

# Build a (channels, days, steps) tensor
matrices = []
for ch in channels_list:
    sub = df[df['channel'] == ch][steps].to_numpy(dtype=float)
    matrices.append(sub)   # each is (90, 3)

tensor = np.stack(matrices, axis=0)  # (5, 90, 3)
print('Tensor shape (channels, days, steps):', tensor.shape)

# Average CVR per channel over all days
cvr_per_channel = tensor[:, :, 2] / tensor[:, :, 0] * 100  # (5, 90)
mean_cvr = cvr_per_channel.mean(axis=1)  # (5,)

for ch, cvr in zip(channels_list, mean_cvr):
    print(f'  {ch:8s}: {cvr:.2f}%')

## Summary
| Operation | Result |
|---|---|
| `arr.reshape(r, c)` | Same data, new shape |
| `arr.reshape(-1)` / `.ravel()` | Flatten to 1D |
| `arr.T` | Transpose (swap axes) |
| `np.concatenate([a,b], axis=0)` | Join along existing axis |
| `np.vstack` / `np.hstack` | Vertical / horizontal join |
| `np.stack([a,b], axis=k)` | Join along NEW axis k |
| `np.split(arr, n)` | Split into n equal pieces |
| `v[:, None]` | 1D → column vector `(n,1)` |

**Next:** `07_linear_algebra.ipynb` — dot products, matrix operations, solving systems.